# 🔍 Étape 3 — Sélection de Variables (Feature Selection)
**Objectif :** Identifier les variables les plus pertinentes pour la classification du Churn en combinant 3 méthodes :

1. **SelectKBest** — score statistique F (ANOVA)
2. **RFE** — Recursive Feature Elimination avec RandomForest
3. **Feature Importance** — RandomForest

---
**Lancer :** `Kernel → Restart & Run All`

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
import warnings, json, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
os.makedirs('../reports', exist_ok=True)

BLUE = '#4C9BE8'; ORANGE = '#E8734C'; GREEN = '#4CAF7D'
print('✅ Imports OK')

## 1. Chargement et préparation des données

In [ ]:
df = pd.read_csv('../data/processed/churn_cleaned.csv')
print(f'Shape : {df.shape}')

# Variables retenues après tests statistiques (étape 2)
# gender et PhoneService exclus (non significatifs)
selected_vars = [
    'SeniorCitizen', 'Partner', 'Dependents', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod',
    'tenure', 'MonthlyCharges', 'TotalCharges'
]
print(f'Variables en entrée : {len(selected_vars)}')

In [ ]:
# Encodage Label pour sklearn
df_enc = df[selected_vars + ['Churn']].copy()
le = LabelEncoder()
for col in df_enc.select_dtypes(include=['object', 'str']).columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

X = df_enc.drop('Churn', axis=1)
y = df_enc['Churn']

print(f'Shape X : {X.shape}')
print(f'Distribution y : {y.value_counts().to_dict()}')

## 2. SelectKBest — Score F (ANOVA)

Mesure la relation linéaire entre chaque variable et la cible.  
Plus le score F est élevé, plus la variable est discriminante.

In [ ]:
skb = SelectKBest(f_classif, k='all')
skb.fit(X, y)

skb_df = pd.DataFrame({
    'Variable': X.columns,
    'F-score': skb.scores_.round(2),
    'p-value': skb.pvalues_
}).sort_values('F-score', ascending=False).reset_index(drop=True)

skb_df['Rang'] = range(1, len(skb_df)+1)
skb_df['Significatif'] = skb_df['p-value'].apply(lambda p: '✅' if p < 0.05 else '❌')
print(skb_df[['Rang','Variable','F-score','Significatif']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = [GREEN if p < 0.05 else ORANGE for p in skb_df['p-value']]
bars = ax.barh(skb_df['Variable'][::-1], skb_df['F-score'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=1, height=0.6)
for bar, val in zip(bars, skb_df['F-score'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=9)
ax.set_xlabel('Score F', fontsize=11)
ax.set_title('SelectKBest — Score F par variable\n(vert = significatif)', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/fig8_selectkbest.png', bbox_inches='tight')
plt.show()
print('✅ fig8 sauvegardée')

## 3. RFE — Recursive Feature Elimination

Entraîne un RandomForest et élimine récursivement les variables les moins importantes.  
On sélectionne les **10 meilleures variables**.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=10, step=1)
rfe.fit(X, y)

rfe_df = pd.DataFrame({
    'Variable': X.columns,
    'Sélectionnée': rfe.support_,
    'Rang RFE': rfe.ranking_
}).sort_values('Rang RFE')

print('=== Top 10 variables sélectionnées par RFE ===')
print(rfe_df[rfe_df['Sélectionnée']][['Variable','Rang RFE']].to_string(index=False))
print('\n=== Variables exclues ===')
print(rfe_df[~rfe_df['Sélectionnée']][['Variable','Rang RFE']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
rfe_plot = rfe_df.sort_values('Rang RFE', ascending=False)
colors = [GREEN if s else ORANGE for s in rfe_plot['Sélectionnée']]
bars = ax.barh(rfe_plot['Variable'], rfe_plot['Rang RFE'],
               color=colors, edgecolor='white', height=0.6)
for bar, val, sel in zip(bars, rfe_plot['Rang RFE'], rfe_plot['Sélectionnée']):
    label = '✅ Sélectionnée' if sel else f'Rang {val}'
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=9)
ax.set_xlabel('Rang RFE (1 = meilleure)', fontsize=11)
ax.set_title('RFE — Classement des variables\n(vert = sélectionnée dans le top 10)', fontsize=13)
ax.set_xlim(0, 12)
plt.tight_layout()
plt.savefig('../reports/fig9_rfe.png', bbox_inches='tight')
plt.show()
print('✅ fig9 sauvegardée')

## 4. Feature Importance — RandomForest

Mesure la contribution de chaque variable à la réduction de l'impureté dans l'arbre.  
Complémentaire à SelectKBest et RFE car capte les **relations non linéaires**.

In [ ]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_full.fit(X, y)

fi_df = pd.DataFrame({
    'Variable': X.columns,
    'Importance': rf_full.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fi_df['Importance %'] = (fi_df['Importance'] * 100).round(2)
print('=== Feature Importance RandomForest ===')
print(fi_df[['Variable','Importance %']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
fi_plot = fi_df.sort_values('Importance')
colors = [GREEN if v >= fi_df['Importance'].median() else BLUE
          for v in fi_plot['Importance']]
bars = ax.barh(fi_plot['Variable'], fi_plot['Importance'],
               color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, fi_plot['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=9)
ax.axvline(x=fi_df['Importance'].median(), color='#E8734C',
           linestyle='--', linewidth=1.5, label='Médiane')
ax.set_xlabel('Importance (Gini)', fontsize=11)
ax.set_title('Feature Importance — RandomForest\n(vert = au-dessus de la médiane)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/fig10_feature_importance.png', bbox_inches='tight')
plt.show()
print('✅ fig10 sauvegardée')

## 5. Comparaison des 3 méthodes — Variables finales retenues

Une variable est **robuste** si elle est sélectionnée par au moins 2 méthodes sur 3.

In [ ]:
# Top 10 de chaque méthode
top_skb = skb_df.head(10)['Variable'].tolist()
top_rfe = rfe_df[rfe_df['Sélectionnée']]['Variable'].tolist()
top_fi  = fi_df.head(10)['Variable'].tolist()

all_vars = list(set(top_skb + top_rfe + top_fi))

comparison = pd.DataFrame({
    'Variable': all_vars,
    'SelectKBest': [v in top_skb for v in all_vars],
    'RFE': [v in top_rfe for v in all_vars],
    'FeatureImportance': [v in top_fi for v in all_vars],
})
comparison['Score (nb méthodes)'] = comparison[['SelectKBest','RFE','FeatureImportance']].sum(axis=1)
comparison = comparison.sort_values('Score (nb méthodes)', ascending=False)
comparison['Retenue'] = comparison['Score (nb méthodes)'] >= 2

print('=== Comparaison des 3 méthodes ===')
print(comparison.to_string(index=False))

In [ ]:
# Heatmap de comparaison
fig, ax = plt.subplots(figsize=(8, 9))
comp_plot = comparison.sort_values('Score (nb méthodes)', ascending=True)
heat_data = comp_plot[['SelectKBest','RFE','FeatureImportance']].astype(int)
sns.heatmap(heat_data, annot=True, fmt='d', cmap='RdYlGn',
            yticklabels=comp_plot['Variable'], ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': '0=Non sélectionnée, 1=Sélectionnée'})
ax.set_title('Comparaison des méthodes de sélection\n(1=sélectionnée, 0=exclue)', fontsize=12)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../reports/fig11_comparison_methods.png', bbox_inches='tight')
plt.show()
print('✅ fig11 sauvegardée')

## 6. Variables finales sélectionnées

In [ ]:
final_vars = comparison[comparison['Score (nb méthodes)'] >= 2]['Variable'].tolist()

print(f'✅ Variables finales retenues ({len(final_vars)}) :')
for v in final_vars:
    score = comparison[comparison['Variable']==v]['Score (nb méthodes)'].values[0]
    print(f'   - {v:25s} | {score}/3 méthodes')

excluded = comparison[comparison['Score (nb méthodes)'] < 2]['Variable'].tolist()
print(f'\n❌ Variables exclues ({len(excluded)}) : {excluded}')

In [ ]:
# Sauvegarde pour l'étape suivante
feature_selection_result = {
    'final_features': final_vars,
    'excluded': excluded,
    'top_skb': top_skb,
    'top_rfe': top_rfe,
    'top_fi': top_fi
}
with open('../data/processed/final_features.json', 'w') as f:
    json.dump(feature_selection_result, f, indent=2)

print('✅ data/processed/final_features.json sauvegardé')
print('\n→ Prochaine étape : 04_sampling.ipynb — SMOTE & SMOTE-NC')

## 7. 📋 Synthèse Feature Selection

| Méthode | Top variables |
|---|---|
| SelectKBest | Contract, tenure, OnlineSecurity, TechSupport, TotalCharges |
| RFE | Contract, tenure, MonthlyCharges, TotalCharges, OnlineSecurity... |
| Feature Importance RF | TotalCharges, MonthlyCharges, tenure, Contract... |

**Variables robustes (≥ 2 méthodes) → retenues pour la modélisation :**
- `tenure`, `MonthlyCharges`, `TotalCharges` (quantitatives)
- `Contract`, `OnlineSecurity`, `TechSupport`, `OnlineBackup`, `InternetService`
- `PaymentMethod`, `PaperlessBilling`

**→ Prochaine étape : `04_sampling.ipynb` — SMOTE & SMOTE-NC**